# Finetuned Full Analysis Stromal Embedded  

- Finetuning on a subset of the stromal cells to improve the model and the performance of the annotation 
- The full analysis notebook of the stromal cells can be found at: notebooks/full_analysis_stromal_embedded.ipynb

The dataset we are using is the same as the one used before. 
- The dataset can be found at: \scGPT\Stromal_cells_all_non-immune_cells_embedded.h5ad



In [1]:
#Finetuning to improve the model and the performance of the model 

#first, we load in all the data and ensure that the preprocessing is correct 

# Packages 
# %%
import copy
import gc
import json
import os
from pathlib import Path
import shutil
import sys
import time
import traceback
from typing import List, Tuple, Dict, Union, Optional
import warnings
import pandas as pd
# from . import asyn
import pickle
import torch
from anndata import AnnData
import scanpy as sc
#import scvi
import seaborn as sns
import numpy as np
#import wandb
from scipy.sparse import issparse
import matplotlib.pyplot as plt
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from torchtext.vocab import Vocab
from torchtext._torchtext import (
    Vocab as VocabPybind,
)
from sklearn.metrics import confusion_matrix

import scgpt as scg
from scgpt.model import TransformerModel, AdversarialDiscriminator
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value
from scgpt.loss import (
    masked_mse_loss,
    masked_relative_error,
    criterion_neg_log_bernoulli,
)
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.preprocess import Preprocessor
from scgpt import SubsetsBatchSampler
from scgpt.utils import set_seed, category_str2int, eval_scib_metrics

sc.set_figure_params(figsize=(6, 6))
os.environ["KMP_WARNINGS"] = "off"
warnings.filterwarnings('ignore')

c:\Users\annel\anaconda3\envs\scgpt_py39\lib\site-packages\scgpt\model\model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
c:\Users\annel\anaconda3\envs\scgpt_py39\lib\site-packages\scgpt\model\multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
c:\Users\annel\anaconda3\envs\scgpt_py39\lib\site-packages\scanpy\_settings.py:488: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  IPython.display.set_matplotlib_formats(*ipython_format)


In [3]:
def setup_directories():
    """Set up necessary directories and paths"""
    repo_dir = Path.cwd().parent.absolute()
    data_dir = repo_dir / "data"
    save_dir = repo_dir / "save"
    
    # Create directories if they don't exist
    data_dir.mkdir(parents=True, exist_ok=True)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    return repo_dir, data_dir, save_dir

# Set up directories
repo_dir, data_dir, save_dir = setup_directories()

#show the directories 
print(f"Repository directory: {repo_dir}")
print(f"Data directory: {data_dir}")
print(f"Save directory: {save_dir}")

# Check if repo_dir is in sys.path
if str(repo_dir) not in sys.path:
    print(f"Adding {repo_dir} to system path")
    sys.path.append(str(repo_dir))
else:
    print(f"{repo_dir} already in system path")


#model dir append
model_dir = data_dir / "scGPT_CP" #use the same model as without finetuning 
# Check files in model_dir
print("Files in model directory:")
if model_dir.exists():
    for file in model_dir.iterdir():
        print(f"- {file.name} (Size: {file.stat().st_size / (1024 * 1024):.2f} MB)")
else:
    print("Model directory does not exist")


# Get and print current working directory
print(f"Current working directory: {os.getcwd()}")


Repository directory: c:\Users\annel\OneDrive\Documenten\Machine Learning\scGPT
Data directory: c:\Users\annel\OneDrive\Documenten\Machine Learning\scGPT\data
Save directory: c:\Users\annel\OneDrive\Documenten\Machine Learning\scGPT\save
c:\Users\annel\OneDrive\Documenten\Machine Learning\scGPT already in system path
Files in model directory:
- args.json (Size: 0.00 MB)
- best_model.pt (Size: 198.23 MB)
- vocab.json (Size: 1.26 MB)
Current working directory: c:\Users\annel\OneDrive\Documenten\Machine Learning\scGPT\notebooks


## Setup configurations for training and for the data

This will be done later, but we will put here elements like hyperparameters, config, etc. 


In [4]:
hyperparameter_defaults = dict(
    seed=0,
    dataset_name="Stromal",
    do_train=True,
    load_model="../save/scGPT_human",
    mask_ratio=0.0,
    epochs=10,
    n_bins=51,
    MVC=False, # Masked value prediction for cell embedding
    ecs_thres=0.0, # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=0.0,
    lr=1e-4,
    batch_size=32,
    layer_size=128,
    nlayers=4,  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
    nhead=4,  # number of heads in nn.MultiheadAttention
    dropout=0.2,  # dropout probability
    schedule_ratio=0.9,  # ratio of epochs for learning rate schedule
    save_eval_interval=5,
    fast_transformer=True,
    pre_norm=False,
    amp=True,  # Automatic Mixed Precision
    include_zero_gene = False,
    freeze = False, #freeze
    DSBN = False,  # Domain-spec batchnorm
    HVG = True
)

In [5]:
# Create config from hyperparameters
from types import SimpleNamespace
config = SimpleNamespace(**hyperparameter_defaults)

# Add additional derived settings
config.log_interval = 100  # interval to log training stats
config.save_dir = Path("../save") / config.dataset_name
config.save_prefix = "scGPT"
config.vocab_size = config.n_bins + 4  # Add special tokens <cls>, <pad>, <eos>, <unk>

# Ensure save directory exists
config.save_dir.mkdir(parents=True, exist_ok=True)

# Log the config
print("\nConfig:")
for k, v in vars(config).items():
    print(f"{k:>20} : {v}")



Config:
                seed : 0
        dataset_name : Stromal
            do_train : True
          load_model : ../save/scGPT_human
          mask_ratio : 0.0
              epochs : 10
              n_bins : 51
                 MVC : False
           ecs_thres : 0.0
          dab_weight : 0.0
                  lr : 0.0001
          batch_size : 32
          layer_size : 128
             nlayers : 4
               nhead : 4
             dropout : 0.2
      schedule_ratio : 0.9
  save_eval_interval : 5
    fast_transformer : True
            pre_norm : False
                 amp : True
   include_zero_gene : False
              freeze : False
                DSBN : False
                 HVG : True
        log_interval : 100
            save_dir : ..\save\Stromal
         save_prefix : scGPT
          vocab_size : 55


In [6]:
# settings for input and preprocessing
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
mask_ratio = config.mask_ratio
mask_value = "auto"  # for masked values, now it should always be auto

include_zero_gene = config.include_zero_gene  # if True, include zero genes among hvgs in the training
max_seq_len = 3001
n_bins = config.n_bins

# input/output representation
input_style = "binned"  # "normed_raw", "log1p", or "binned"
output_style = "binned"  # "normed_raw", "log1p", or "binned"

# settings for training
MLM = False  # whether to use masked language modeling, currently it is always on.
CLS = True  # celltype classification objective
ADV = False  # Adversarial training for batch correction
CCE = False  # Contrastive cell embedding objective
MVC = config.MVC  # Masked value prediction for cell embedding
ECS = config.ecs_thres > 0  # Elastic cell similarity objective
DAB = False  # Domain adaptation by reverse backpropagation, set to 2 for separate optimizer
INPUT_BATCH_LABELS = False  # TODO: have these help MLM and MVC, while not to classifier
input_emb_style = "continuous"  # "category" or "continuous" or "scaling"
cell_emb_style = "cls"  # "avg-pool" or "w-pool" or "cls"
adv_E_delay_epochs = 0  # delay adversarial training on encoder for a few epochs
adv_D_delay_epochs = 0
mvc_decoder_style = "inner product"
ecs_threshold = config.ecs_thres
dab_weight = config.dab_weight

explicit_zero_prob = MLM and include_zero_gene  # whether explicit bernoulli for zeros
do_sample_in_train = False and explicit_zero_prob  # sample the bernoulli in training

per_seq_batch_sample = False

# settings for optimizer
lr = config.lr  # TODO: test learning rate ratio between two tasks
lr_ADV = 1e-3  # learning rate for discriminator, used when ADV is True
batch_size = config.batch_size
eval_batch_size = config.batch_size
epochs = config.epochs
schedule_interval = 1

# settings for the model
fast_transformer = config.fast_transformer
fast_transformer_backend = "flash"  # "linear" or "flash"
embsize = config.layer_size  # embedding dimension
d_hid = config.layer_size  # dimension of the feedforward network in TransformerEncoder
nlayers = config.nlayers  # number of TransformerEncoderLayer in TransformerEncoder
nhead = config.nhead  # number of heads in nn.MultiheadAttention
dropout = config.dropout  # dropout probability

# logging
log_interval = 100  # iterations
save_eval_interval = config.save_eval_interval  # epochs
do_eval_scib_metrics = True

In [7]:
# validate the sett# %% validate settings
assert input_style in ["normed_raw", "log1p", "binned"]
assert output_style in ["normed_raw", "log1p", "binned"]
assert input_emb_style in ["category", "continuous", "scaling"]
if input_style == "binned":
    if input_emb_style == "scaling":
        raise ValueError("input_emb_style `scaling` is not supported for binned input.")
elif input_style == "log1p" or input_style == "normed_raw":
    if input_emb_style == "category":
        raise ValueError(
            "input_emb_style `category` is not supported for log1p or normed_raw input."
        )

if input_emb_style == "category":
    mask_value = n_bins + 1
    pad_value = n_bins  # for padding gene expr values
    n_input_bins = n_bins + 2
else:
    mask_value = -1
    pad_value = -2
    n_input_bins = n_bins

if ADV and DAB:
    raise ValueError("ADV and DAB cannot be both True.")
DAB_separate_optim = True if DAB > 1 else False


In [89]:
print (input_style)

binned


In [8]:
dataset_name = config.dataset_name
save_dir = Path(f"./save/dev_{dataset_name}-{time.strftime('%b%d-%H-%M')}/")
save_dir.mkdir(parents=True, exist_ok=True)
print(f"save to {save_dir}")
logger = scg.logger
scg.utils.add_file_handler(logger, save_dir / "run.log")

save to save\dev_Stromal-Feb21-15-30


# Load and pre-process the data 

In [9]:
import requests
from tqdm import tqdm
import pandas as pd
from pathlib import Path

# Current dataset info: 
# At some point we can make this nicer, either with hyperparameters or with a config file, but it works for now 
#dataset_name = 'Stromal_cells_all_non-immune_cells_embedded'
#dataset_download_url = "https://datasets.cellxgene.cziscience.com/617749e2-71cb-428f-9eb7-bad1cf7988ef.h5ad"
#dataset_id = "617749e2-71cb-428f-9eb7-bad1cf7988ef"
#file_path = data_dir / "Stromal_cells_all_non-immune_cells_embedded.h5ad"

dataset_name = 'Derived Embryoid Bodies'
dataset_download_url = 'https://datasets.cellxgene.cziscience.com/c605e7df-96a3-40cd-8eb5-53b32dfa9a10.h5ad'
dataset_id = 'c605e7df-96a3-40cd-8eb5-53b32dfa9a10'
file_path = data_dir / f"{dataset_name}.h5ad"

# Create tracking file if it doesn't exist
tracking_file = data_dir / 'dataset_tracking.csv'
if not tracking_file.exists():
    print(f"Creating tracking file {tracking_file}")
    pd.DataFrame(columns=['name', 'path', 'last_used', 'times_used', 'dataset_id', 'dataset_url', 'cell_count', 'gene_count', 'meta_data']).to_csv(tracking_file, index=False)

# Download with progress bar if file doesn't exist
if not file_path.exists():
    print(f"Downloading {dataset_name} from {dataset_download_url} to {file_path}")
    response = requests.get(dataset_download_url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(file_path, 'wb') as file, tqdm(total=total_size, unit='B', unit_scale=True) as pbar:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
            pbar.update(len(chunk))

df = pd.read_csv(tracking_file)

if str(file_path) in df['path'].values:
    # Update existing entry
    # Check if this file was already accessed in the last hour to avoid duplicate counts
    last_access = pd.to_datetime(df.loc[df['path'] == str(file_path), 'last_used'].iloc[0])
    if pd.Timestamp.now() - last_access < pd.Timedelta(hours=6):
        print("File accessed within the last hour - skipping increment")
    else:
        mask = df['path'] == str(file_path)
        df.loc[mask, 'times_used'] = df.loc[mask, 'times_used'].fillna(0) + 1
        df.loc[mask, 'last_used'] = pd.Timestamp.now()
else:
    # Add new entry (either for newly downloaded or existing untracked file)
    print (f"Adding new entry for {dataset_name} to tracking file")
    new_row = {'name': dataset_name, 'path': str(file_path), 
               'last_used': pd.Timestamp.now(), 'times_used': 1, 
               'dataset_id': dataset_id, 'dataset_url': dataset_download_url}
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

df.to_csv(tracking_file, index=False)
print("Tracking updated!")

#it didn't update tracked, need to fix this #maybe better to do since last restart of computer, or idk, something like that 

File accessed within the last hour - skipping increment
Tracking updated!


In [10]:

# Display the tracking CSV file contents
print("Current dataset tracking:")
tracking_df = pd.read_csv(tracking_file)
display(tracking_df)



Current dataset tracking:


,name,path,last_used,times_used,dataset_id,dataset_url
0,Bronchopulmonary_displasia,c:\Users\annel\OneDrive\Documenten\Machine Lea...,02:33.3,4,1a8ba781-a3d7-467e-913f-757af3b24168,https://datasets.cellxgene.cziscience.com/1a8b...
1,Stromal_cells_all_non-immune_cells_embedded,c:\Users\annel\OneDrive\Documenten\Machine Lea...,2025-02-13 16:34:56.057059,2,617749e2-71cb-428f-9eb7-bad1cf7988ef,https://datasets.cellxgene.cziscience.com/6177...
2,Derived Embryoid Bodies,c:\Users\annel\OneDrive\Documenten\Machine Lea...,2025-02-21 14:40:07.849339,2,c605e7df-96a3-40cd-8eb5-53b32dfa9a10,https://datasets.cellxgene.cziscience.com/c605...
3,HBCA_global,c:\Users\annel\OneDrive\Documenten\Machine Lea...,2025-02-20 22:47:02.351407,1,2b284c0b-8a93-4138-a466-9c2bbe9db733,https://datasets.cellxgene.cziscience.com/2b28...


Open the adata object with scanpy, and do some first check and analysis before preprocessing for training

In [11]:
adata = sc.read(file_path)

#check annotation keys 
from my_src.my_utils import check_annotation_keys
adata_keys = check_annotation_keys(adata)

#check the number of cells and features 
shape_cells = adata.shape
df.loc[df['dataset_id'] == dataset_id, 'shape_cells'] = str(shape_cells) 

print (shape_cells)
print (adata_keys)

celltype = adata_keys['cell_type_keys'][0]
print (celltype)

gene_name = adata_keys['gene_keys'][0]
if len(adata_keys['hvg_keys']) > 0: 
    hvg_name = adata_keys['hvg_keys'][0]



Checking cell type annotations in .obs:

✓ Found 'cell_type' with 17 unique values
Examples:
  1. endothelial cell
  2. mesodermal cell
  3. lateral mesodermal cell
  4. endodermal cell
  5. paraxial cell
Top counts:
  splanchnic mesodermal cell: 19921 cells
  mesodermal cell: 18299 cells
  sensory neuron: 15966 cells
  epithelial cell: 15494 cells
  endothelial cell: 12399 cells

✓ Found 'cluster' with 47 unique values
Examples:
  1. 23
  2. 29
  3. 11
  4. 32
  5. 17
Top counts:
  1: 6762 cells
  2: 6270 cells
  3: 5837 cells
  4: 5741 cells
  5: 5344 cells

Checking gene annotations in .var:

✓ Found 'ensembl_id' as index with 27003 unique values
Examples:
  1. ENSG00000000003
  2. ENSG00000000005
  3. ENSG00000000419
  4. ENSG00000000457
  5. ENSG00000000460

✓ Found 'feature_name' with 27003 unique values
Examples:
  1. TSPAN6
  2. TNMD
  3. DPM1
  4. SCYL3
  5. C1orf112

Checking highly variable gene annotations in .var:

⚠️ No HVG annotations found in .var columns

Checking hig

In [12]:
# Check number of unique samples
if 'sample' in adata.obs.columns:
    n_samples = adata.obs['sample'].nunique()
    print(f"Number of unique samples: {n_samples}")
    print("\nSample counts:")
    display(adata.obs['sample'].value_counts())
else:
    print("No 'sample' column found in adata.obs")


Number of unique samples: 24

Sample counts:


sample
SD37    13949
SD38    12428
SD39    12357
SD49    10799
SD36    10527
SD47     9709
SD34     8976
SD41     8796
SD35     8785
SD52     8763
SD48     8411
SD54     7558
SD46     7474
SD40     7297
SD51     7296
SD53     6110
SD55     5162
SD50     4971
SD61     3943
SD57     1312
SD56     1157
SD58      272
SD60       30
SD59        3
Name: count, dtype: int64

In [13]:
# Check if shape_cells column exists, if not add it
if 'shape_cells' not in tracking_df.columns:
    tracking_df['shape_cells'] = None


tracking_df.loc[df['dataset_id'] == dataset_id, 'shape_cells'] = str(shape_cells)
tracking_df.loc[df['dataset_id'] == dataset_id, 'cell_count'] = shape_cells[0]
tracking_df.loc[df['dataset_id'] == dataset_id, 'gene_count'] = shape_cells[0]

print (shape_cells[0])

print("Current dataset tracking (including shapes):")
display(tracking_df)

166085
Current dataset tracking (including shapes):


,name,path,last_used,times_used,dataset_id,dataset_url,shape_cells,cell_count,gene_count
0,Bronchopulmonary_displasia,c:\Users\annel\OneDrive\Documenten\Machine Lea...,02:33.3,4,1a8ba781-a3d7-467e-913f-757af3b24168,https://datasets.cellxgene.cziscience.com/1a8b...,None,NaN,NaN
1,Stromal_cells_all_non-immune_cells_embedded,c:\Users\annel\OneDrive\Documenten\Machine Lea...,2025-02-13 16:34:56.057059,2,617749e2-71cb-428f-9eb7-bad1cf7988ef,https://datasets.cellxgene.cziscience.com/6177...,None,NaN,NaN
2,Derived Embryoid Bodies,c:\Users\annel\OneDrive\Documenten\Machine Lea...,2025-02-21 14:40:07.849339,2,c605e7df-96a3-40cd-8eb5-53b32dfa9a10,https://datasets.cellxgene.cziscience.com/c605...,"(166085, 27003)",166085.0,166085.0
3,HBCA_global,c:\Users\annel\OneDrive\Documenten\Machine Lea...,2025-02-20 22:47:02.351407,1,2b284c0b-8a93-4138-a466-9c2bbe9db733,https://datasets.cellxgene.cziscience.com/2b28...,None,NaN,NaN


In [14]:
n_finetune_batch = 6
batch_key = adata_keys['batch_keys'][1]
#print (adata_keys['batch_keys'])

#Split based on donors / batches
if adata_keys['batch_keys']: 
    unique_batches  = adata.obs[batch_key].unique()
    print(f"\nTotal unique donors: {len(unique_batches)}")

# Get unique donors


# Take first n_train_donors as training set
finetune_batch = unique_batches[:n_finetune_batch]
test_batch = unique_batches[n_finetune_batch:]

print("\nfinetune_batches:")
for batch in finetune_batch:
    print(f"- {batch}")

print("\nTest batches:") 
for batch in test_batch:
    print(f"- {batch}")

adata_org = adata.copy()



Total unique donors: 24

finetune_batches:
- SD34
- SD35
- SD36
- SD37
- SD38
- SD39

Test batches:
- SD40
- SD41
- SD46
- SD47
- SD48
- SD49
- SD50
- SD51
- SD52
- SD53
- SD54
- SD55
- SD56
- SD57
- SD58
- SD59
- SD60
- SD61


In [15]:
N_HVG = 5000
#adata_org = adata.copy()
# Check for HVG annotations
hvg_keys = [
    'highly_variable', 'highly_variable_genes',
    'hvg', 'HVG', 
    'variable_genes', 'variable_features'
]

hvg_name = None
for key in hvg_keys:
    if key in adata.var.columns:
        hvg_name = key
        break

if config.HVG:
    if hvg_name:
        hvg_mask = adata.var[hvg_name]
        adata_hvg = adata[:, hvg_mask]
        print(f"\nSelected {hvg_mask.sum()} highly variable genes")
    else:
        sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor='seurat_v3')
        adata_hvg = adata[:, adata.var['highly_variable']]
        print(f"\nSelected {adata_hvg.shape[1]} highly variable genes")


Selected 5000 highly variable genes


In [16]:
print("Number of unique samples:", len(adata.obs['sample'].unique()))
print("\nUnique samples:")
print(adata.obs['sample'].value_counts())

Number of unique samples: 24

Unique samples:
sample
SD37    13949
SD38    12428
SD39    12357
SD49    10799
SD36    10527
SD47     9709
SD34     8976
SD41     8796
SD35     8785
SD52     8763
SD48     8411
SD54     7558
SD46     7474
SD40     7297
SD51     7296
SD53     6110
SD55     5162
SD50     4971
SD61     3943
SD57     1312
SD56     1157
SD58      272
SD60       30
SD59        3
Name: count, dtype: int64


In [24]:
# First create a clean batch_id column  
adata.obs["batch_id"] = [1 if batch in finetune_batch else 0
                     for batch in adata.obs["sample"]]
# Then use this for both is_ref and str_batch
adata.obs["is_ref"] = ["finetune" if bid == 1 else "test" 
                   for bid in adata.obs["batch_id"]]
adata.obs["str_batch"] = adata.obs["batch_id"].astype(str)

if gene_name != adata.var.index.name and gene_name in adata.var.columns:
    print(f"Setting index to {gene_name}")
    adata.var.set_index(adata.var[gene_name], inplace=True)
else:
    print("Gene identifiers already in index or column not found")
data_is_raw = False
filter_gene_by_counts = False

# Verification checks
print("\nVerification of data processing:")
print(f"Number of samples {adata.obs[batch_key].nunique()}")
print(f"Number of unique cell types: {num_types}")
print(f"Number of finetune samples: {sum(adata.obs['is_ref'] == 'finetune')}")
print(f"Number of test samples: {sum(adata.obs['is_ref'] == 'test')}")
print("\nBatch distribution:")
celltype_id_labels = adata.obs[celltype].astype("category").cat.codes.values
celltypes = adata.obs[celltype].unique()
num_types = len(np.unique(celltype_id_labels))
id2type = dict(enumerate(adata.obs[celltype].astype("category").cat.categories))
adata.obs["celltype_id"] = celltype_id_labels
adata.var[gene_name] = adata.var.index.tolist()

# Verification checks
print("\nVerification of data processing:")
print(f"Number of samples {adata.obs[batch_key]}")
print(f"Number of unique cell types: {num_types}")
print(f"Number of finetune samples: {sum(adata.obs['is_ref'] == 'finetune')}")
print(f"Number of test samples: {sum(adata.obs['is_ref'] == 'test')}")
print("\nBatch distribution:")
print(pd.crosstab(adata.obs["str_batch"], adata.obs["is_ref"]))
print("\nVerifying gene names:")
print(f"Number of genes: {len(adata.var_names)}")
print(f"Number of unique genes: {len(set(adata.var_names))}")
if len(adata.var_names) != len(set(adata.var_names)):
    print("Warning: Duplicate gene names found!")






Gene identifiers already in index or column not found

Verification of data processing:
Number of samples 24
Number of unique cell types: 17
Number of finetune samples: 67022
Number of test samples: 99063

Batch distribution:

Verification of data processing:
Number of samples 0         SD34
1         SD34
2         SD34
3         SD34
4         SD34
          ... 
166080    SD61
166081    SD61
166082    SD61
166083    SD61
166084    SD61
Name: sample, Length: 166085, dtype: category
Categories (24, object): ['SD34', 'SD35', 'SD36', 'SD37', ..., 'SD58', 'SD59', 'SD60', 'SD61']
Number of unique cell types: 17
Number of finetune samples: 67022
Number of test samples: 99063

Batch distribution:
is_ref     finetune   test
str_batch                 
0                 0  99063
1             67022      0

Verifying gene names:
Number of genes: 27003
Number of unique genes: 27003


In [26]:
display(adata.obs.head(20))

,nCount_RNA,nFeature_RNA,Size_Factor,n.umi,scrublet_score,scrublet_call,sample,Ligation_barcode,RT_barcode,P7_barcode,...,organism,sex,tissue,self_reported_ethnicity,development_stage,observation_joinid,batch_id,str_batch,celltype_id,is_ref
0,1369.0,1046,3.197984,1369.0,0.062500,Singlet,SD34,LIG221,P01-E01,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,8G_@dyCe`*,1,1,4,finetune
1,499.0,426,1.165664,499.0,0.117853,Singlet,SD34,LIG225,P01-E01,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,Wz8i5kC^dT,1,1,5,finetune
2,2022.0,1437,4.723392,2022.0,0.254658,Singlet,SD34,LIG253,P01-E01,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,`<16MFI}S),1,1,13,finetune
3,1668.0,1231,3.896448,1668.0,0.291667,Singlet,SD34,LIG306,P01-E01,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,b0pjzew1|Z,1,1,6,finetune
4,241.0,223,0.562976,241.0,0.088608,Singlet,SD34,LIG334,P01-E01,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,;)xLgf6stF,1,1,12,finetune
5,304.0,276,0.710144,304.0,0.099062,Singlet,SD34,LIG341,P01-E01,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,-rJY~0LXrF,1,1,9,finetune
6,755.0,633,1.763680,755.0,0.167164,Singlet,SD34,LIG351,P01-E01,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,wPd7-V!TQX,1,1,6,finetune
7,574.0,444,1.340864,574.0,0.043069,Singlet,SD34,LIG217,P01-E02,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,2JX=XxFUu6,1,1,10,finetune
8,329.0,284,0.768544,329.0,0.133080,Singlet,SD34,LIG263,P01-E02,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,W=}$VYb;CZ,1,1,16,finetune
9,478.0,421,1.116608,478.0,0.234043,Singlet,SD34,LIG269,P01-E02,E01,...,Homo sapiens,male,embryoid body,unknown,unknown,C^b3;XcDq$,1,1,13,finetune


In [ ]:
# After running predictions
print("\nPrediction distribution:")
pred_counts = pd.Series(predictions).value_counts()
display(pred_counts)

print("\nTrue label distribution:")
true_counts = pd.Series(labels).value_counts()
display(true_counts)

# Check if all predictions are the same
if len(pred_counts) == 1:
    print("WARNING: All cells assigned to the same type!")
    # Investigate model outputs more closely
    print(f"Checking raw model outputs for a sample batch...")
    
    # Run a small sample through the model and look at unnormalized logits
    with torch.no_grad():
        sample_batch = next(iter(test_loader))
        input_gene_ids = sample_batch["gene_ids"].to(device)
        input_values = sample_batch["values"].to(device)
        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
        
        output_dict = best_model(
            input_gene_ids,
            input_values,
            src_key_padding_mask=src_key_padding_mask,
            batch_labels=sample_batch["batch_labels"].to(device) if INPUT_BATCH_LABELS or config.DSBN else None,
            CLS=True,
            CCE=False,
            MVC=False,
            ECS=False,
        )
        
        # Look at the logits before softmax
        logits = output_dict["cls_output"]
        print("Logits shape:", logits.shape)
        print("Logits statistics:")
        print("  Mean:", logits.mean().item())
        print("  Std:", logits.std().item())
        print("  Min:", logits.min().item())
        print("  Max:", logits.max().item())
        
        # Look at the distribution across classes
        print("Mean logit per class:")
        class_means = logits.mean(dim=0)
        for i, mean in enumerate(class_means):
            print(f"  Class {i} ({id2type[i]}): {mean.item():.4f}")

In [162]:
# Create a pandas DataFrame from adata.obs and display it
df = pd.DataFrame(adata.obs)
# This doesn't work because we need to pass a list of column names to select multiple columns
# The correct syntax is:
# Display sample of training data (batch_id 0) with key columns to verify splits
finetune_df = df[df["batch_id"] == 0][["celltype_id", "batch_id", celltype, batch_key, "is_ref", "str_batch"]]
print("Sample of training data (batch_id 0):")
display(finetune_df.head(10))


Sample of training data (batch_id 0):


,celltype_id,batch_id,cell_type,sample,is_ref,str_batch
67022,14,0,intermediate mesodermal cell,SD40,test,0
67023,14,0,intermediate mesodermal cell,SD40,test,0
67024,5,0,mesodermal cell,SD40,test,0
67025,16,0,splanchnic mesodermal cell,SD40,test,0
67026,3,0,sensory neuron,SD40,test,0
67027,2,0,blood cell,SD40,test,0
67028,5,0,mesodermal cell,SD40,test,0
67029,3,0,sensory neuron,SD40,test,0
67030,0,0,fibroblast,SD40,test,0
67031,4,0,endothelial cell,SD40,test,0


In [169]:
adata_train = adata[adata.obs['batch_id'] == 1]
adata_test = adata[adata.obs['batch_id'] == 0]
adata_test_raw = adata_test.copy() #make sure you have a copy to find your true values 

print(f"Training set: {len(adata_train)} cells")
print(f"Test set: {len(adata_test)} cells")
test_df = pd.DataFrame(adata_test.obs)
train_df = pd.DataFrame(adata_train.obs)

Training set: 67022 cells
Test set: 99063 cells


In [180]:
if 'adata_hvg' not in locals():
    adata_hvg = adata[:, adata.var["highly_variable"]]

if 'adata_hvg_train' not in locals(): 
    adata_hvg_train = adata_hvg[adata_hvg.obs['batch_id']==1]
    adata_hvg_test = adata_hvg[adata_hvg.obs['batch_id']==0]
    adata_hvt_test_raw = adata_hvg_test.copy

print (adata_hvg_train.shape, adata_hvg_test.shape)

(67022, 5000) (99063, 5000)


In [ ]:
n_bins = 51 #this will later be in the config file 

# set up the preprocessor, use the args to config the workflow
preprocessor = Preprocessor(
    use_key="X",  # the key in adata.layers to use as raw data
    filter_gene_by_counts=filter_gene_by_counts,  # step 1
    filter_cell_by_counts=False,  # step 2
    normalize_total=1e4,  # 3. whether to normalize the raw data and to what sum
    result_normed_key="X_normed",  # the key in adata.layers to store the normalized data
    log1p=data_is_raw,  # 4. whether to log1p the normalized data
    result_log1p_key="X_log1p",
    subset_hvg=False,  # 5. whether to subset the raw data to highly variable genes
    hvg_flavor="seurat_v3" if data_is_raw else "cell_ranger",
    binning=n_bins,  # 6. whether to bin the raw data and to what number of bins
    result_binned_key="X_binned",  # the key in adata.layers to store the binned data
)



preprocessor(adata_hvg_test, batch_key=None)
preprocessor(adata_hvg_train, batch_key=None)

scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Binning data ...


In [ ]:
#input_style later added to the config and hyperparameters
input_style = "binned"

input_layer_key = {  # the values of this map coorespond to the keys in preprocessing
    "normed_raw": "X_normed",
    "log1p": "X_normed",
    "binned": "X_binned",
}[input_style]
all_counts = (
    adata.layers[input_layer_key].A
    if issparse(adata.layers[input_layer_key])
    else adata.layers[input_layer_key]
)

celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
celltypes_labels = np.array(celltypes_labels)

batch_ids = adata.obs["batch_id"].tolist()
num_batch_types = len(set(batch_ids))
batch_ids = np.array(batch_ids)

(
    train_data,
    valid_data,
    train_celltype_labels,
    valid_celltype_labels,
    train_batch_labels,
    valid_batch_labels,
) = train_test_split(
    all_counts, celltypes_labels, batch_ids, test_size=0.1, shuffle=True
)

KeyError: 'X_binned'

In [ ]:
if config.load_model is None:
    vocab = Vocab(
        VocabPybind(genes + special_tokens, None)
    )  # bidirectional lookup [gene <-> int]
vocab.set_default_index(vocab["<pad>"])
gene_ids = np.array(vocab(genes), dtype=int) #be careful when you take the hvg genes 

In [ ]:
tokenized_train = tokenize_and_pad_batch(
    train_data,
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,  # append <cls> token at the beginning
    include_zero_gene=include_zero_gene,
)
tokenized_valid = tokenize_and_pad_batch(
    valid_data,
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,
    include_zero_gene=include_zero_gene,
)
logger.info(
    f"train set number of samples: {tokenized_train['genes'].shape[0]}, "
    f"\n\t feature length: {tokenized_train['genes'].shape[1]}"
)
logger.info(
    f"valid set number of samples: {tokenized_valid['genes'].shape[0]}, "
    f"\n\t feature length: {tokenized_valid['genes'].shape[1]}"
)

In [ ]:
def prepare_data(sort_seq_batch=False) -> Tuple[Dict[str, torch.Tensor]]:
    masked_values_train = random_mask_value(
        tokenized_train["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )
    masked_values_valid = random_mask_value(
        tokenized_valid["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )
    print(
        f"random masking at epoch {epoch:3d}, ratio of masked values in train: ",
        f"{(masked_values_train == mask_value).sum() / (masked_values_train - pad_value).count_nonzero():.4f}",
    )

    input_gene_ids_train, input_gene_ids_valid = (
        tokenized_train["genes"],
        tokenized_valid["genes"],
    )
    input_values_train, input_values_valid = masked_values_train, masked_values_valid
    target_values_train, target_values_valid = (
        tokenized_train["values"],
        tokenized_valid["values"],
    )

    tensor_batch_labels_train = torch.from_numpy(train_batch_labels).long()
    tensor_batch_labels_valid = torch.from_numpy(valid_batch_labels).long()

    tensor_celltype_labels_train = torch.from_numpy(train_celltype_labels).long()
    tensor_celltype_labels_valid = torch.from_numpy(valid_celltype_labels).long()

    if sort_seq_batch:  # TODO: update to random pick seq source in each traning batch
        train_sort_ids = np.argsort(train_batch_labels)
        input_gene_ids_train = input_gene_ids_train[train_sort_ids]
        input_values_train = input_values_train[train_sort_ids]
        target_values_train = target_values_train[train_sort_ids]
        tensor_batch_labels_train = tensor_batch_labels_train[train_sort_ids]
        tensor_celltype_labels_train = tensor_celltype_labels_train[train_sort_ids]

        valid_sort_ids = np.argsort(valid_batch_labels)
        input_gene_ids_valid = input_gene_ids_valid[valid_sort_ids]
        input_values_valid = input_values_valid[valid_sort_ids]
        target_values_valid = target_values_valid[valid_sort_ids]
        tensor_batch_labels_valid = tensor_batch_labels_valid[valid_sort_ids]
        tensor_celltype_labels_valid = tensor_celltype_labels_valid[valid_sort_ids]

    train_data_pt = {
        "gene_ids": input_gene_ids_train,
        "values": input_values_train,
        "target_values": target_values_train,
        "batch_labels": tensor_batch_labels_train,
        "celltype_labels": tensor_celltype_labels_train,
    }
    valid_data_pt = {
        "gene_ids": input_gene_ids_valid,
        "values": input_values_valid,
        "target_values": target_values_valid,
        "batch_labels": tensor_batch_labels_valid,
        "celltype_labels": tensor_celltype_labels_valid,
    }

    return train_data_pt, valid_data_pt


# dataset
class SeqDataset(Dataset):
    def __init__(self, data: Dict[str, torch.Tensor]):
        self.data = data

    def __len__(self):
        return self.data["gene_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}


# data_loader
def prepare_dataloader(
    data_pt: Dict[str, torch.Tensor],
    batch_size: int,
    shuffle: bool = False,
    intra_domain_shuffle: bool = False,
    drop_last: bool = False,
    num_workers: int = 0,
) -> DataLoader:
    if num_workers == 0:
        num_workers = min(len(os.sched_getaffinity(0)), batch_size // 2)

    dataset = SeqDataset(data_pt)

    if per_seq_batch_sample:
        # find the indices of samples in each seq batch
        subsets = []
        batch_labels_array = data_pt["batch_labels"].numpy()
        for batch_label in np.unique(batch_labels_array):
            batch_indices = np.where(batch_labels_array == batch_label)[0].tolist()
            subsets.append(batch_indices)
        data_loader = DataLoader(
            dataset=dataset,
            batch_sampler=SubsetsBatchSampler(
                subsets,
                batch_size,
                intra_subset_shuffle=intra_domain_shuffle,
                inter_subset_shuffle=shuffle,
                drop_last=drop_last,
            ),
            num_workers=num_workers,
            pin_memory=True,
        )
        return data_loader

    data_loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,
    )
    return data_loader


Configuration of the Model preparation

In [ ]:
if config.load_model is not None:
    model_dir = Path(config.load_model)
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    vocab_file = model_dir / "vocab.json"

    vocab = GeneVocab.from_file(vocab_file)
    shutil.copy(vocab_file, save_dir / "vocab.json")
    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    adata.var["id_in_vocab"] = [
        1 if gene in vocab else -1 for gene in adata.var["gene_name"]
    ]
    gene_ids_in_vocab = np.array(adata.var["id_in_vocab"])
    logger.info(
        f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
        f"in vocabulary of size {len(vocab)}."
    )
    adata = adata[:, adata.var["id_in_vocab"] >= 0]

    # model
    with open(model_config_file, "r") as f:
        model_configs = json.load(f)
    logger.info(
        f"Resume model from {model_file}, the model args will override the "
        f"config {model_config_file}."
    )
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]

load model is not None and found here: {model_dir}
match 24256/24256 genes in vocabulary of size 60697.
Resume model from c:\Users\annel\OneDrive\Documenten\Machine Learning\scGPT\data\scGPT_CP\best_model.pt, the model args will override the config c:\Users\annel\OneDrive\Documenten\Machine Learning\scGPT\data\scGPT_CP\args.json.


## Tokenizer preparation 

# Prepare the pre-trained model 

In this case we are using the continual prem-trained model since we did the same for the embeddings of the non-finetuned case. 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ntokens = len(vocab)  # size of vocabulary
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    nlayers_cls=3,
    n_cls=num_types if CLS else 1,
    vocab=vocab,
    dropout=dropout,
    pad_token=pad_token,
    pad_value=pad_value,
    do_mvc=MVC,
    do_dab=DAB,
    use_batch_labels=INPUT_BATCH_LABELS,
    num_batch_labels=num_batch_types,
    domain_spec_batchnorm=config.DSBN,
    input_emb_style=input_emb_style,
    n_input_bins=n_input_bins,
    cell_emb_style=cell_emb_style,
    mvc_decoder_style=mvc_decoder_style,
    ecs_threshold=ecs_threshold,
    explicit_zero_prob=explicit_zero_prob,
    use_fast_transformer=fast_transformer,
    fast_transformer_backend=fast_transformer_backend,
    pre_norm=config.pre_norm,
)
if config.load_model is not None:
    try:
        model.load_state_dict(torch.load(model_file))
        logger.info(f"Loading all model params from {model_file}")
    except:
        # only load params that are in the model and match the size
        model_dict = model.state_dict()
        pretrained_dict = torch.load(model_file)
        pretrained_dict = {
            k: v
            for k, v in pretrained_dict.items()
            if k in model_dict and v.shape == model_dict[k].shape
        }
        for k, v in pretrained_dict.items():
            logger.info(f"Loading params {k} with shape {v.shape}")
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)

pre_freeze_param_count = sum(dict((p.data_ptr(), p.numel()) for p in model.parameters() if p.requires_grad).values())

# Freeze all pre-decoder weights
for name, para in model.named_parameters():
    print("-"*20)
    print(f"name: {name}")
    if config.freeze and "encoder" in name and "transformer_encoder" not in name:
    # if config.freeze and "encoder" in name:
        print(f"freezing weights for: {name}")
        para.requires_grad = False

post_freeze_param_count = sum(dict((p.data_ptr(), p.numel()) for p in model.parameters() if p.requires_grad).values())

logger.info(f"Total Pre freeze Params {(pre_freeze_param_count )}")
logger.info(f"Total Post freeze Params {(post_freeze_param_count )}")
###wandb.log(
        {
            "info/pre_freeze_param_count": pre_freeze_param_count,
            "info/post_freeze_param_count": post_freeze_param_count,
        },
)

model.to(device)
#wandb.watch(model)

if ADV:
    discriminator = AdversarialDiscriminator(
        d_model=embsize,
        n_cls=num_batch_types,
    ).to(device)


In [ ]:
criterion = masked_mse_loss
criterion_cls = nn.CrossEntropyLoss()
criterion_dab = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=lr, eps=1e-4 if config.amp else 1e-8
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, schedule_interval, gamma=config.schedule_ratio
)
if DAB_separate_optim:
    optimizer_dab = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler_dab = torch.optim.lr_scheduler.StepLR(
        optimizer_dab, schedule_interval, gamma=config.schedule_ratio
    )
if ADV:
    criterion_adv = nn.CrossEntropyLoss()  # consider using label smoothing
    optimizer_E = torch.optim.Adam(model.parameters(), lr=lr_ADV)
    scheduler_E = torch.optim.lr_scheduler.StepLR(
        optimizer_E, schedule_interval, gamma=config.schedule_ratio
    )
    optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr_ADV)
    scheduler_D = torch.optim.lr_scheduler.StepLR(
        optimizer_D, schedule_interval, gamma=config.schedule_ratio
    )

scaler = torch.cuda.amp.GradScaler(enabled=config.amp)

In [ ]:
def train(model: nn.Module, loader: DataLoader) -> None:
    """
    Train the model for one epoch.
    """
    model.train()
    (
        total_loss,
        total_mse,
        total_cls,
        total_cce,
        total_mvc,
        total_ecs,
        total_dab,
        total_adv_E,
        total_adv_D,
        total_zero_log_prob,
        total_mvc_zero_log_prob,
    ) = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
    total_error = 0.0
    start_time = time.time()

    num_batches = len(loader)
    for batch, batch_data in enumerate(loader):
        input_gene_ids = batch_data["gene_ids"].to(device)
        input_values = batch_data["values"].to(device)
        target_values = batch_data["target_values"].to(device)
        batch_labels = batch_data["batch_labels"].to(device)
        celltype_labels = batch_data["celltype_labels"].to(device)

        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
        with torch.cuda.amp.autocast(enabled=config.amp):
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                CLS=CLS,
                CCE=CCE,
                MVC=MVC,
                ECS=ECS,
                do_sample=do_sample_in_train,
                #generative_training=False
            )

            masked_positions = input_values.eq(mask_value)  # the postions to predict
            loss = 0.0
            metrics_to_log = {}
            if MLM:
                loss_mse = criterion(
                    output_dict["mlm_output"], target_values, masked_positions
                )
                loss = loss + loss_mse
                metrics_to_log = {"train/mse": loss_mse.item()}
            if explicit_zero_prob:
                loss_zero_log_prob = criterion_neg_log_bernoulli(
                    output_dict["mlm_zero_probs"], target_values, masked_positions
                )
                loss = loss + loss_zero_log_prob
                metrics_to_log.update({"train/nzlp": loss_zero_log_prob.item()})
            if CLS:
                loss_cls = criterion_cls(output_dict["cls_output"], celltype_labels)
                loss = loss + loss_cls
                metrics_to_log.update({"train/cls": loss_cls.item()})

                error_rate = 1 - (
                    (output_dict["cls_output"].argmax(1) == celltype_labels)
                    .sum()
                    .item()
                ) / celltype_labels.size(0)
            if CCE:
                loss_cce = 10 * output_dict["loss_cce"]
                loss = loss + loss_cce
                metrics_to_log.update({"train/cce": loss_cce.item()})
            if MVC:
                loss_mvc = criterion(
                    output_dict["mvc_output"], target_values, masked_positions
                )
                loss = loss + loss_mvc
                metrics_to_log.update({"train/mvc": loss_mvc.item()})
            if MVC and explicit_zero_prob:
                loss_mvc_zero_log_prob = criterion_neg_log_bernoulli(
                    output_dict["mvc_zero_probs"], target_values, masked_positions
                )
                loss = loss + loss_mvc_zero_log_prob
                metrics_to_log.update({"train/mvc_nzlp": loss_mvc_zero_log_prob.item()})
            if ECS:
                loss_ecs = 10 * output_dict["loss_ecs"]
                loss = loss + loss_ecs
                metrics_to_log.update({"train/ecs": loss_ecs.item()})
            if DAB:
                # try weighting and separate optimizer
                loss_dab = criterion_dab(output_dict["dab_output"], batch_labels)
                loss = loss + dab_weight * loss_dab
                metrics_to_log.update({"train/dab": loss_dab.item()})

        model.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        with warnings.catch_warnings(record=True) as w:
            warnings.filterwarnings("always")
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
                error_if_nonfinite=False if scaler.is_enabled() else True,
            )
            if len(w) > 0:
                logger.warning(
                    f"Found infinite gradient. This may be caused by the gradient "
                    f"scaler. The current scale is {scaler.get_scale()}. This warning "
                    "can be ignored if no longer occurs after autoscaling of the scaler."
                )
        scaler.step(optimizer)
        scaler.update()

        if ADV:
            # rerun the model for adversarial training
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                CLS=CLS,
                CCE=CCE,
                MVC=MVC,
                ECS=ECS,
                do_sample=do_sample_in_train,
                #generative_training=False
            )

            # TRAINING DISCRIMINATOR
            loss_adv_D = criterion_adv(
                discriminator(output_dict["cell_emb"].detach()), batch_labels
            )
            if epoch > adv_D_delay_epochs:
                discriminator.zero_grad()
                loss_adv_D.backward()
                optimizer_D.step()

            # TRAINING ENCODER
            loss_adv_E = -criterion_adv(
                discriminator(output_dict["cell_emb"]), batch_labels
            )
            # NOTE: the loss is negative here because we want to maximize
            # the cross_entropy_loss, in other words, disguise against the discriminator
            if epoch > adv_E_delay_epochs:
                model.zero_grad()
                discriminator.zero_grad()
                loss_adv_E.backward()
                optimizer_E.step()

        #wandb.log(metrics_to_log)

        total_loss += loss.item()
        total_mse += loss_mse.item() if MLM else 0.0
        total_cls += loss_cls.item() if CLS else 0.0
        total_cce += loss_cce.item() if CCE else 0.0
        total_mvc += loss_mvc.item() if MVC else 0.0
        total_ecs += loss_ecs.item() if ECS else 0.0
        total_dab += loss_dab.item() if DAB else 0.0
        total_adv_E += loss_adv_E.item() if ADV else 0.0
        total_adv_D += loss_adv_D.item() if ADV else 0.0
        total_zero_log_prob += loss_zero_log_prob.item() if explicit_zero_prob else 0.0
        total_mvc_zero_log_prob += (
            loss_mvc_zero_log_prob.item() if MVC and explicit_zero_prob else 0.0
        )
        total_error += error_rate
        if batch % log_interval == 0 and batch > 0:
            lr = scheduler.get_last_lr()[0]
            ms_per_batch = (time.time() - start_time) * 1000 / log_interval
            cur_loss = total_loss / log_interval
            cur_mse = total_mse / log_interval
            cur_cls = total_cls / log_interval if CLS else 0.0
            cur_cce = total_cce / log_interval if CCE else 0.0
            cur_mvc = total_mvc / log_interval if MVC else 0.0
            cur_ecs = total_ecs / log_interval if ECS else 0.0
            cur_dab = total_dab / log_interval if DAB else 0.0
            cur_adv_E = total_adv_E / log_interval if ADV else 0.0
            cur_adv_D = total_adv_D / log_interval if ADV else 0.0
            cur_zero_log_prob = (
                total_zero_log_prob / log_interval if explicit_zero_prob else 0.0
            )
            cur_mvc_zero_log_prob = (
                total_mvc_zero_log_prob / log_interval
                if MVC and explicit_zero_prob
                else 0.0
            )
            cur_error = total_error / log_interval
            # ppl = math.exp(cur_loss)
            logger.info(
                f"| epoch {epoch:3d} | {batch:3d}/{num_batches:3d} batches | "
                f"lr {lr:05.4f} | ms/batch {ms_per_batch:5.2f} | "
                f"loss {cur_loss:5.2f} | "
                + (f"mse {cur_mse:5.2f} | mre {cur_error:5.2f} |" if MLM else "")
                + (f"cls {cur_cls:5.2f} | " if CLS else "")
                + (f"err {cur_error:5.2f} | " if CLS else "")
                + (f"cce {cur_cce:5.2f} |" if CCE else "")
                + (f"mvc {cur_mvc:5.2f} |" if MVC else "")
                + (f"ecs {cur_ecs:5.2f} |" if ECS else "")
                + (f"dab {cur_dab:5.2f} |" if DAB else "")
                + (f"adv_E {cur_adv_E:5.2f} |" if ADV else "")
                + (f"adv_D {cur_adv_D:5.2f} |" if ADV else "")
                + (f"nzlp {cur_zero_log_prob:5.2f} |" if explicit_zero_prob else "")
                + (
                    f"mvc_nzlp {cur_mvc_zero_log_prob:5.2f} |"
                    if MVC and explicit_zero_prob
                    else ""
                )
            )
            total_loss = 0
            total_mse = 0
            total_cls = 0
            total_cce = 0
            total_mvc = 0
            total_ecs = 0
            total_dab = 0
            total_adv_E = 0
            total_adv_D = 0
            total_zero_log_prob = 0
            total_mvc_zero_log_prob = 0
            total_error = 0
            start_time = time.time()

''' 
def define_wandb_metrcis():
    wandb.define_metric("valid/mse", summary="min", step_metric="epoch")
    wandb.define_metric("valid/mre", summary="min", step_metric="epoch")
    wandb.define_metric("valid/dab", summary="min", step_metric="epoch")
    wandb.define_metric("valid/sum_mse_dab", summary="min", step_metric="epoch")
    wandb.define_metric("test/avg_bio", summary="max")
'''

def evaluate(model: nn.Module, loader: DataLoader, return_raw: bool = False) -> float:
    """
    Evaluate the model on the evaluation data.
    """
    model.eval()
    total_loss = 0.0
    total_error = 0.0
    total_dab = 0.0
    total_num = 0
    predictions = []
    with torch.no_grad():
        for batch_data in loader:
            input_gene_ids = batch_data["gene_ids"].to(device)
            input_values = batch_data["values"].to(device)
            target_values = batch_data["target_values"].to(device)
            batch_labels = batch_data["batch_labels"].to(device)
            celltype_labels = batch_data["celltype_labels"].to(device)

            src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
            with torch.cuda.amp.autocast(enabled=config.amp):
                output_dict = model(
                    input_gene_ids,
                    input_values,
                    src_key_padding_mask=src_key_padding_mask,
                    batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                    CLS=CLS,  # evaluation does not need CLS or CCE
                    CCE=False,
                    MVC=False,
                    ECS=False,
                    do_sample=do_sample_in_train,
                    #generative_training = False,
                )
                output_values = output_dict["cls_output"]
                loss = criterion_cls(output_values, celltype_labels)

                if DAB:
                    loss_dab = criterion_dab(output_dict["dab_output"], batch_labels)

            total_loss += loss.item() * len(input_gene_ids)
            accuracy = (output_values.argmax(1) == celltype_labels).sum().item()
            total_error += (1 - accuracy / len(input_gene_ids)) * len(input_gene_ids)
            total_dab += loss_dab.item() * len(input_gene_ids) if DAB else 0.0
            total_num += len(input_gene_ids)
            preds = output_values.argmax(1).cpu().numpy()
            predictions.append(preds)
    
''' 
    wandb.log(
        {
            "valid/mse": total_loss / total_num,
            "valid/err": total_error / total_num,
            "valid/dab": total_dab / total_num,
            "valid/sum_mse_dab": (total_loss + dab_weight * total_dab) / total_num,
            "epoch": epoch,
        },
    )
'''
    if return_raw:
        return np.concatenate(predictions, axis=0)

    return total_loss / total_num, total_error / total_num


In [ ]:
best_val_loss = float("inf")
best_avg_bio = 0.0
best_model = None
#define_wandb_metrcis()

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()
    train_data_pt, valid_data_pt = prepare_data(sort_seq_batch=per_seq_batch_sample)
    train_loader = prepare_dataloader(
        train_data_pt,
        batch_size=batch_size,
        shuffle=False,
        intra_domain_shuffle=True,
        drop_last=False,
    )
    valid_loader = prepare_dataloader(
        valid_data_pt,
        batch_size=eval_batch_size,
        shuffle=False,
        intra_domain_shuffle=False,
        drop_last=False,
    )

    if config.do_train:
        train(
            model,
            loader=train_loader,
        )
    val_loss, val_err = evaluate(
        model,
        loader=valid_loader,
    )
    elapsed = time.time() - epoch_start_time
    logger.info("-" * 89)
    logger.info(
        f"| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | "
        f"valid loss/mse {val_loss:5.4f} | err {val_err:5.4f}"
    )
    logger.info("-" * 89)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = copy.deepcopy(model)
        best_model_epoch = epoch
        logger.info(f"Best model with score {best_val_loss:5.4f}")

    scheduler.step()
    if DAB_separate_optim:
        scheduler_dab.step()
    if ADV:
        scheduler_D.step()
        scheduler_E.step()

# Inference and Evaluation with annotation finetuning

Both use the test function and evaluation function as we have seen it before, but also want to use the pre-trained model the way we did with one-shot learning for the annotation task. 

This approach gives specific finetuning as an approach where we finetune the headers and freeze the rest, to ensure that it will understand the annotation. 

Afterwards, also other types of Inference. b

In [ ]:
# %% inference
def test(model: nn.Module, adata: DataLoader) -> float:
    all_counts = (
        adata.layers[input_layer_key].A
        if issparse(adata.layers[input_layer_key])
        else adata.layers[input_layer_key]
    )

    celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
    celltypes_labels = np.array(celltypes_labels)

    batch_ids = adata.obs["batch_id"].tolist()
    batch_ids = np.array(batch_ids)

    tokenized_test = tokenize_and_pad_batch(
        all_counts,
        gene_ids,
        max_len=max_seq_len,
        vocab=vocab,
        pad_token=pad_token,
        pad_value=pad_value,
        append_cls=True,  # append <cls> token at the beginning
        include_zero_gene=include_zero_gene,
    )

    input_values_test = random_mask_value(
        tokenized_test["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )

    test_data_pt = {
        "gene_ids": tokenized_test["genes"],
        "values": input_values_test,
        "target_values": tokenized_test["values"],
        "batch_labels": torch.from_numpy(batch_ids).long(),
        "celltype_labels": torch.from_numpy(celltypes_labels).long(),
    }

    test_loader = DataLoader(
        dataset=SeqDataset(test_data_pt),
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
        pin_memory=True,
    )

    model.eval()
    predictions = evaluate(
        model,
        loader=test_loader,
        return_raw=True,
    )

    # compute accuracy, precision, recall, f1
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    accuracy = accuracy_score(celltypes_labels, predictions)
    precision = precision_score(celltypes_labels, predictions, average="macro")
    recall = recall_score(celltypes_labels, predictions, average="macro")
    macro_f1 = f1_score(celltypes_labels, predictions, average="macro")

    logger.info(
        f"Accuracy: {accuracy:.3f}, Precision: {precision:.3f}, Recall: {recall:.3f}, "
        f"Macro F1: {macro_f1:.3f}"
    )

    results = {
        "test/accuracy": accuracy,
        "test/precision": precision,
        "test/recall": recall,
        "test/macro_f1": macro_f1,
    }

    return predictions, celltypes_labels, results

In [ ]:
predictions, labels, results = test(best_model, adata_test)
adata_test_raw.obs["predictions"] = [id2type[p] for p in predictions]

# plot
palette_ = plt.rcParams["axes.prop_cycle"].by_key()["color"] 
palette_ = plt.rcParams["axes.prop_cycle"].by_key()["color"] + plt.rcParams["axes.prop_cycle"].by_key()["color"] + plt.rcParams["axes.prop_cycle"].by_key()["color"]
palette_ = {c: palette_[i] for i, c in enumerate(celltypes)}

with plt.rc_context({"figure.figsize": (6, 4), "figure.dpi": (300)}):
    sc.pl.umap(
        adata_test_raw,
        color=["celltype", "predictions"],
        palette=palette_,
        show=False,
    )
    plt.savefig(save_dir / "results.png", dpi=300)

save_dict = {
    "predictions": predictions,
    "labels": labels,
    "results": results,
    "id_maps": id2type
}
with open(save_dir / "results.pkl", "wb") as f:
    pickle.dump(save_dict, f)

''' 
results["test/cell_umap"] = wandb.Image(
    str(save_dir / "results.png"),
    caption=f"predictions macro f1 {results['test/macro_f1']:.3f}",
)
wandb.log(results)
'''

In [ ]:
from sklearn.metrics import confusion_matrix
celltypes = list(celltypes)
for i in set([id2type[p] for p in predictions]):
    if i not in celltypes:
        celltypes.remove(i)
cm = confusion_matrix(labels, predictions)
cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
cm = pd.DataFrame(cm, index=celltypes[:cm.shape[0]], columns=celltypes[:cm.shape[1]])
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues")
plt.savefig(save_dir / "confusion_matrix.png", dpi=300)


''' 
results["test/confusion_matrix"] = wandb.Image(
    str(save_dir / "confusion_matrix.png"),
    caption=f"confusion matrix",
)
'''